# Hybrid Biomedical IR — Full Pipeline (Master Notebook)

CAP 6776 Information Retrieval. **Runtime → Run All** executes the complete, real pipeline in order: dataset audit → TF-IDF → BM25 → BGE → MedCPT → Hybrid RRF → Cross-Encoder Reranking → Statistical Analysis → Error Analysis → Web Export.

This notebook calls `scripts/reproduce.py`, which skips any step whose output artifact already exists (pass `force=True` below to recompute everything). Individual per-stage notebooks (`00`–`11`) cover each step with more explanation; this one is the single-cell reproduction path.

**⚠️ Expect this to take 30–60+ minutes** end-to-end on a fresh environment (BGE/MedCPT encoding + cross-encoder reranking across 3 candidate pool sizes dominate the runtime) — see each stage notebook for per-step timing measured on Apple M1 Pro (MPS).

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
force = False  # set True to recompute every step, ignoring cached results/ artifacts

cmd = [sys.executable, 'scripts/reproduce.py']
if force:
    cmd.append('--force')

result = subprocess.run(cmd, cwd=os.getcwd())
assert result.returncode == 0, 'Pipeline failed -- see output above for which step and why.'

## Final results table

In [ ]:
import json
import pandas as pd

models = ['tfidf', 'bm25', 'bge', 'medcpt', 'hybrid_rrf', 'hybrid_reranked']
rows = []
for m in models:
    d = json.load(open(f'results/metrics/{m}.json'))
    metrics = d['metrics']
    rows.append({'model': m, 'P@10': metrics['P@10'], 'Recall@100': metrics['Recall@100'],
                 'MAP': metrics['MAP'], 'MRR@10': metrics['MRR@10'], 'nDCG@10': metrics['nDCG@10']})
pd.DataFrame(rows).set_index('model')

See the project README, `paper/results.md`, and `results/tables/` for the full writeup, statistical tests, and error analysis.